# Twin-Prime-Compatible Residue Transition Experiment

**Prime Numbers Lab**

This notebook tracks residue transitions modulo \(30\) compatible with twin primes:

\[
11 \to 13,\qquad 17 \to 19,\qquad 29 \to 1.
\]

Outputs:

```text
figures/twin_prime_transition_entries.png
figures/twin_prime_transition_entries.csv
figures/twin_prime_transition_operator.png
figures/twin_prime_transition_experiment_outputs.zip
```

The notebook works inside the repo and in Colab. It uses `data/primes.npy` if available, with a `sympy` fallback.

## 1. Setup

In [ ]:

from pathlib import Path
import sys
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path(".")
FIG_DIR = ROOT / "figures"
DATA_DIR = ROOT / "data"

FIG_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29])
RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}

TWIN_TRANSITIONS = [
    (11, 13),
    (17, 19),
    (29, 1),
]

plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

print("Figure directory:", FIG_DIR.resolve())
print("Data directory:", DATA_DIR.resolve())

## 2. Load transition utilities

In [ ]:

try:
    from src.transitions import primes_mod_30, build_transition_matrix, RESIDUES
    RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}
    print("Loaded transition utilities from src.transitions")
except Exception as e:
    print("Using local fallback transition utilities:", repr(e))

    RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29])
    RES_IDX = {int(r): i for i, r in enumerate(RESIDUES)}

    def primes_mod_30(primes):
        return np.array([int(p) % 30 for p in primes if int(p) > 5])

    def build_transition_matrix(seq):
        n = len(RESIDUES)
        P = np.zeros((n, n), dtype=float)

        for i in range(len(seq) - 1):
            a, b = int(seq[i]), int(seq[i + 1])
            if a in RES_IDX and b in RES_IDX:
                P[RES_IDX[a], RES_IDX[b]] += 1

        row_sums = P.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        return P / row_sums

## 3. Load primes

In [ ]:

def load_primes(path=DATA_DIR / "primes.npy", fallback_limit=2_000_000):
    path = Path(path)

    if path.exists():
        print(f"Loading primes from {path}")
        primes = np.load(path)
        print(f"Loaded {len(primes):,} primes.")
        return primes

    print(f"{path} not found.")
    print(f"Generating primes up to {fallback_limit:,} using sympy fallback...")

    try:
        from sympy import primerange
    except ImportError as exc:
        raise ImportError(
            "sympy is required for fallback prime generation. "
            "Install with `pip install sympy`, or provide data/primes.npy."
        ) from exc

    primes = np.array(list(primerange(2, fallback_limit)), dtype=np.int64)
    print(f"Generated {len(primes):,} primes.")
    return primes

primes = load_primes()
print("First primes:", primes[:10])
print("Last primes:", primes[-5:])

## 4. Compute twin-prime-compatible entries across sample sizes

In [ ]:

def sample_sizes(n_total):
    candidates = [
        1_000,
        3_000,
        10_000,
        30_000,
        100_000,
        300_000,
        1_000_000,
        3_000_000,
        10_000_000,
    ]
    sizes = [n for n in candidates if n <= n_total]
    if not sizes:
        sizes = [n_total]
    return sizes

def transition_entry(P, src, dst):
    return P[RES_IDX[int(src)], RES_IDX[int(dst)]]

sizes = sample_sizes(len(primes))
print("Sample sizes:", sizes)

results = {f"{a}->{b}": [] for a, b in TWIN_TRANSITIONS}

for n in sizes:
    seq = primes_mod_30(primes[:n])
    P = build_transition_matrix(seq)

    for a, b in TWIN_TRANSITIONS:
        results[f"{a}->{b}"].append(transition_entry(P, a, b))

df = pd.DataFrame({"N": sizes, **results})
df

## 5. Plot tracked transition entries

In [ ]:

fig, ax = plt.subplots(figsize=(8, 5))

for label in results:
    ax.plot(df["N"], df[label], marker="o", linewidth=2, label=label)

ax.set_xscale("log")
ax.set_xlabel("Number of primes used")
ax.set_ylabel("Transition probability")
ax.set_title("Twin-prime-compatible residue transitions modulo 30")
ax.legend(title="Transition")
fig.tight_layout()

png_path = FIG_DIR / "twin_prime_transition_entries.png"
fig.savefig(png_path, dpi=220)
plt.show()

print("Saved:", png_path)

## 6. Save CSV output

In [ ]:

csv_path = FIG_DIR / "twin_prime_transition_entries.csv"
df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df

## 7. Plot final transition operator with twin-prime entries highlighted

In [ ]:

seq = primes_mod_30(primes[:sizes[-1]])
P = build_transition_matrix(seq)

fig, ax = plt.subplots(figsize=(6.8, 5.8))
im = ax.imshow(P, cmap="viridis")

ax.set_xticks(range(len(RESIDUES)))
ax.set_yticks(range(len(RESIDUES)))
ax.set_xticklabels(RESIDUES)
ax.set_yticklabels(RESIDUES)

ax.set_xlabel("Next residue")
ax.set_ylabel("Current residue")
ax.set_title(f"Transition operator P with twin-prime-compatible entries, N={sizes[-1]:,}")

for src, dst in TWIN_TRANSITIONS:
    y = RES_IDX[src]
    x = RES_IDX[dst]
    ax.scatter([x], [y], s=260, facecolors="none", edgecolors="white", linewidths=2.5)
    ax.text(x, y, "★", ha="center", va="center", color="white", fontsize=13)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Transition probability")

fig.tight_layout()

operator_path = FIG_DIR / "twin_prime_transition_operator.png"
fig.savefig(operator_path, dpi=220)
plt.show()

print("Saved:", operator_path)

## 8. Interpretation

The tracked transitions are exactly the residue transitions modulo \(30\) compatible with twin primes:

\[
11 \to 13,\qquad 17 \to 19,\qquad 29 \to 1.
\]

This experiment does not prove the twin prime conjecture.

It measures how twin-prime-compatible residue transitions appear in the transition operator \(P\), and how those entries change as the number of primes \(N\) increases.

## 9. Package outputs as a zip

In [ ]:

zip_path = FIG_DIR / "twin_prime_transition_experiment_outputs.zip"

output_files = [
    FIG_DIR / "twin_prime_transition_entries.png",
    FIG_DIR / "twin_prime_transition_entries.csv",
    FIG_DIR / "twin_prime_transition_operator.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in output_files:
        if path.exists():
            z.write(path, arcname=path.name)

print("Saved:", zip_path)
print("Contents:")
with zipfile.ZipFile(zip_path, "r") as z:
    for name in z.namelist():
        print(" -", name)

## 10. Optional Colab download

In [ ]:

try:
    from google.colab import files
    files.download(str(FIG_DIR / "twin_prime_transition_experiment_outputs.zip"))
except ImportError:
    print("Download only works in Google Colab.")
    print("Zip saved locally at:")
    print(FIG_DIR / "twin_prime_transition_experiment_outputs.zip")